## Environment Setup
Load environment variables from `.env` and configure LangSmith with the EU endpoint.
- `LANGSMITH_TRACING`: enables automatic tracing of LLM calls
- `LANGSMITH_ENDPOINT`: uses the EU server
- `LANGSMITH_PROJECT`: the project name where experiments will be logged

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "arduino-qa-eval"

## Initialize Clients
Initialize LangSmith and OpenAI clients. `wrap_openai` adds automatic
tracing to every OpenAI call.

In [ ]:
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

client = Client()
openai_client = wrap_openai(OpenAI())
print("Clients ready!")

## Define Arduino Q&A Dataset
15 Arduino Q&A examples, each with a question (input) and a reference answer (output).

In [ ]:
examples = [
    {"question": "What is the purpose of the setup() function in Arduino?", "answer": "The setup() function runs once when the Arduino powers on or resets. It is used to initialize variables, pin modes, and libraries."},
    {"question": "What is the difference between digitalRead() and analogRead()?", "answer": "digitalRead() reads a digital pin and returns HIGH or LOW. analogRead() reads an analog pin and returns a value between 0 and 1023."},
    {"question": "What does pinMode() do?", "answer": "pinMode() configures a specific pin as either INPUT, OUTPUT, or INPUT_PULLUP."},
    {"question": "What is the loop() function in Arduino?", "answer": "The loop() function runs repeatedly after setup(). It contains the main logic of the program that executes continuously."},
    {"question": "How do you turn on an LED connected to pin 13?", "answer": "Use pinMode(13, OUTPUT) in setup(), then digitalWrite(13, HIGH) in loop() to turn it on."},
    {"question": "What is PWM in Arduino?", "answer": "PWM (Pulse Width Modulation) simulates analog output using digital pins. It is used with analogWrite() on PWM-capable pins to control things like LED brightness or motor speed."},
    {"question": "What is the difference between delay() and millis()?", "answer": "delay() pauses the program for a given number of milliseconds, blocking all execution. millis() returns the time elapsed since the program started and allows non-blocking timing."},
    {"question": "What voltage does a standard Arduino Uno operate at?", "answer": "The Arduino Uno operates at 5V logic level."},
    {"question": "What is a serial monitor used for in Arduino?", "answer": "The serial monitor is used to send and receive text data between the Arduino and a computer, useful for debugging and displaying sensor values."},
    {"question": "How do you read a button press on Arduino?", "answer": "Connect the button to a digital pin, use pinMode(pin, INPUT_PULLUP), then use digitalRead(pin) to check if the value is LOW when the button is pressed."},
    {"question": "What is the difference between int and long in Arduino?", "answer": "int stores 16-bit integers (-32768 to 32767). long stores 32-bit integers (-2,147,483,648 to 2,147,483,647), used when larger numbers are needed."},
    {"question": "What does the Wire library do in Arduino?", "answer": "The Wire library enables I2C communication between the Arduino and other I2C-compatible devices like sensors and displays."},
    {"question": "What is a servo motor and how do you control it with Arduino?", "answer": "A servo motor is a motor that rotates to a specific angle. It is controlled using the Servo library with myServo.write(angle) where angle is between 0 and 180."},
    {"question": "What is the purpose of a pull-up resistor in Arduino circuits?", "answer": "A pull-up resistor ensures a digital pin reads HIGH by default when no signal is applied, preventing floating pin states."},
    {"question": "How do you store data permanently on an Arduino?", "answer": "Use the EEPROM library to read and write data to the Arduino's built-in EEPROM memory, which persists after power off."},
]

print(f"Total examples: {len(examples)}")

## Create LangSmith Dataset
Upload the examples to LangSmith as a reusable dataset.
Checks if the dataset already exists before creating it.

In [ ]:
dataset_name = "Arduino Q&A Evaluation Dataset"

existing = [d.name for d in client.list_datasets()]
if dataset_name not in existing:
    dataset = client.create_dataset(
        dataset_name,
        description="15 Arduino programming Q&A examples for LLM evaluation."
    )
    client.create_examples(
        inputs=[{"question": e["question"]} for e in examples],
        outputs=[{"answer": e["answer"]} for e in examples],
        dataset_id=dataset.id,
    )
    print(f"Created dataset with {len(examples)} examples!")
else:
    print(f"Dataset '{dataset_name}' already exists — skipping upload.")

## Target Function
Takes a question from the dataset, sends it to GPT-4o-mini, and returns
the answer. The @traceable decorator logs every call to LangSmith automatically.

In [ ]:
@traceable
def answer_arduino_question(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an expert Arduino programming assistant. Answer questions clearly and concisely."},
            {"role": "user", "content": inputs["question"]}
        ]
    )
    return {"answer": response.choices[0].message.content.strip()}

# Quick test
result = answer_arduino_question({"question": "What is the purpose of the setup() function in Arduino?"})
print(result["answer"])

## Set Up Correctness Evaluator
LLM-as-judge using GPT-4o-mini as the judge model with the built-in
CORRECTNESS_PROMPT from openevals. Wrapped in a function matching
LangSmith's expected evaluator signature.

In [ ]:
_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge=openai_client,
    model="gpt-4o-mini",
    feedback_key="correctness",
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)

print("Evaluator ready!")

## Run Evaluation
Run the full evaluation on the dataset using the target function
and correctness evaluator. Results are automatically logged to LangSmith.

In [ ]:
results = client.evaluate(
    answer_arduino_question,
    data="Arduino Q&A Evaluation Dataset",
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-mini",
    max_concurrency=2,
)
print(results)

## Inspect Results Structure
Check the structure of the results object to find the correct field names.

In [ ]:
# Inspect the first result
r = results._results[0]
print("=== KEYS ===")
print(r.keys())
print()
print("=== RUN INPUTS ===")
print(r["run"].inputs)
print()
print("=== RUN OUTPUTS ===")
print(r["run"].outputs)
print()
print("=== EXAMPLE OUTPUTS ===")
print(r["example"].outputs)
print()
print("=== EVALUATION RESULTS ===")
print(r["evaluation_results"])

## Analyze Results
Extract scores from the evaluation results and compute summary statistics.

In [ ]:
import pandas as pd

rows = []
for r in results._results:
    rows.append({
        "question": r["run"].inputs["inputs"]["question"],
        "answer": r["run"].outputs["answer"],
        "reference": r["example"].outputs["answer"],
        "correctness": r["evaluation_results"]["results"][0].score,
        "comment": r["evaluation_results"]["results"][0].comment,
    })

df = pd.DataFrame(rows)

# Summary statistics
print("=== EVALUATION SUMMARY ===")
print(f"Total examples:    {len(df)}")
print(f"Correct (True):    {df['correctness'].sum()}")
print(f"Incorrect (False): {(~df['correctness']).sum()}")
print(f"Pass rate:         {df['correctness'].mean()*100:.1f}%")
print()
print("=== SCORES PER QUESTION ===")
for _, row in df.iterrows():
    status = "✅" if row["correctness"] else "❌"
    print(f"{status} {row['question'][:70]}")

## Evaluation Report
A summary report of the evaluation process and findings.

In [ ]:
report = """
# Evaluation Report: Arduino Q&A with GPT-4o-mini

## Executive Summary
GPT-4o-mini was evaluated on a custom Arduino programming Q&A dataset of 15 examples.
The model achieved a 100% correctness rate, correctly answering all questions when
judged against reference answers using an LLM-as-judge evaluator.

## Methodology

### Dataset
- **Domain**: Arduino programming
- **Size**: 15 examples
- **Source**: Manually curated Q&A pairs covering core Arduino concepts
- **Format**: Each example contains a question (input) and a reference answer (output)
- **Topics covered**: pin modes, digital/analog I/O, PWM, timing functions, 
  serial communication, libraries (Wire, Servo, EEPROM), data types, and circuits

### Target Function
- **Model**: GPT-4o-mini (temperature=0)
- **System prompt**: "You are an expert Arduino programming assistant. 
  Answer questions clearly and concisely."
- **Tracing**: Enabled via LangSmith `@traceable` decorator and `wrap_openai`

### Evaluator
- **Type**: LLM-as-judge using CORRECTNESS_PROMPT from openevals
- **Judge model**: GPT-4o-mini
- **Scoring**: Binary (True/False)
- **Criteria**: Factual accuracy and semantic equivalence with reference answer

## Results
| Metric            | Value  |
|-------------------|--------|
| Total examples    | 15     |
| Correct answers   | 15     |
| Incorrect answers | 0      |
| Pass rate         | 100%   |

All 15 questions were judged as correct. The model consistently provided accurate,
complete answers that matched or exceeded the reference answers in detail.

## Analysis

### Strengths
- Perfect factual accuracy across all Arduino topics
- Answers were often more detailed than reference answers while remaining correct
- Consistent performance across different question types (concepts, code, circuits)

### Limitations
- Dataset is small (15 examples) — results may not generalise to harder questions
- Reference answers were manually written and may not cover all valid responses
- Binary scoring (True/False) does not capture partial correctness or answer quality
- The judge model (GPT-4o-mini) is the same as the target model, which may introduce
  self-preference bias in scoring

### Observations
- The model tended to give longer, more detailed answers than the reference
- The LLM judge correctly recognised semantic equivalence despite phrasing differences
- Topics like PWM, I2C, and EEPROM were handled accurately despite being more advanced

## Recommendations
1. **Expand the dataset** with harder, more ambiguous questions to stress-test the model
2. **Use a different judge model** (e.g. GPT-4o) to avoid self-preference bias
3. **Add custom evaluators** to measure answer conciseness and format quality
4. **Test with a weaker model** (e.g. GPT-3.5) to create a meaningful performance baseline
"""

print(report)